In [1]:
from huggingface_hub import login
login()

## IMPORTS

In [3]:
import os
import re
import time
import tempfile
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from faster_whisper import WhisperModel
from jiwer import wer, cer

## CONFIG

In [4]:
HF_DATASET_NAME = "google/fleurs"
HF_CONFIG_NAME = "id_id"      # Indonesian
HF_SPLIT = "test"             # final benchmark split

MODEL_NAME = "cahya/faster-whisper-medium-id"

DEVICE = "cpu"                # "cpu" or "cuda"
COMPUTE_TYPE = "int8"         # cpu: int8 | gpu: float16 / int8_float16

MAX_SAMPLES = None            # contoh: 10 buat test cepat, None untuk semua
BEAM_SIZE = 5
USE_VAD = True

DETAIL_CSV = "hasil_evaluasi_fleurs_id_detail.csv"
SUMMARY_CSV = "hasil_evaluasi_fleurs_id_summary.csv"

## TEXT CLEANING

In [5]:
def clean_text(text: str) -> str:
    text = str(text).lower().strip()

    # hapus isi kurung siku dan tag
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"<[^>]*>", " ", text)

    # ganti beberapa separator dengan spasi
    text = text.replace("-", " ").replace("/", " ")

    # sisakan huruf/angka/spasi
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # rapikan spasi
    text = re.sub(r"\s+", " ", text).strip()
    return text

## LOAD DATASET

In [6]:
print("Loading FLEURS dataset...")
ds = load_dataset(HF_DATASET_NAME, HF_CONFIG_NAME, split=HF_SPLIT)

print(ds)
print("\nContoh field sample:")
print(ds[0].keys())

# FLEURS docs list these fields across splits:
# id, num_samples, path, audio, raw_transcription, transcription, gender, lang_id, lang_group_id

if MAX_SAMPLES is not None:
    ds = ds.select(range(min(MAX_SAMPLES, len(ds))))

print(f"\nJumlah sample yang dievaluasi: {len(ds)}")


Loading FLEURS dataset...
Dataset({
    features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
    num_rows: 687
})

Contoh field sample:
dict_keys(['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'])

Jumlah sample yang dievaluasi: 687


## LOAD MODEL

In [7]:
model = WhisperModel(
    MODEL_NAME,
    device=DEVICE,
    compute_type=COMPUTE_TYPE
)
print("Model loaded.")

Model loaded.


## EVALUATION LOOP

In [8]:
detail_rows = []

references_raw = []
predictions_raw = []

references_clean = []
predictions_clean = []

latencies = []
num_failed = 0

for sample in tqdm(ds, total=len(ds), desc="Evaluating FLEURS id_id"):
    sample_id = sample["id"]
    audio_obj = sample["audio"]
    ref_raw = sample["transcription"]           # normalized transcript from dataset
    ref_raw_alt = sample.get("raw_transcription", "")

    # audio field is a dict with array + sampling_rate + path
    audio_array = audio_obj["array"]
    sampling_rate = audio_obj["sampling_rate"]

    tmp_wav_path = None
    try:
        # faster-whisper works smoothly with file paths, so write temp wav
        import soundfile as sf

        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_wav:
            tmp_wav_path = tmp_wav.name
            sf.write(tmp_wav_path, audio_array, sampling_rate)

        start = time.time()
        segments, info = model.transcribe(
            tmp_wav_path,
            language="id",
            beam_size=BEAM_SIZE,
            vad_filter=USE_VAD
        )
        elapsed = time.time() - start

        pred_raw = " ".join(seg.text.strip() for seg in segments).strip()

        ref_clean = clean_text(ref_raw)
        pred_clean = clean_text(pred_raw)

        references_raw.append(ref_raw)
        predictions_raw.append(pred_raw)

        references_clean.append(ref_clean)
        predictions_clean.append(pred_clean)

        latencies.append(elapsed)

        detail_rows.append({
            "id": sample_id,
            "hf_split": HF_SPLIT,
            "reference_text_raw": ref_raw,
            "reference_text_alt_raw": ref_raw_alt,
            "pred_text_raw": pred_raw,
            "reference_text_clean": ref_clean,
            "pred_text_clean": pred_clean,
            "latency_sec": elapsed,
            "detected_language": info.language,
            "language_probability": info.language_probability,
            "sampling_rate": sampling_rate,
            "status": "ok"
        })

    except Exception as e:
        num_failed += 1
        detail_rows.append({
            "id": sample_id,
            "hf_split": HF_SPLIT,
            "reference_text_raw": ref_raw,
            "reference_text_alt_raw": ref_raw_alt,
            "pred_text_raw": "",
            "reference_text_clean": clean_text(ref_raw),
            "pred_text_clean": "",
            "latency_sec": None,
            "detected_language": None,
            "language_probability": None,
            "sampling_rate": sampling_rate,
            "status": f"failed: {type(e).__name__}: {str(e)}"
        })
    finally:
        if tmp_wav_path and os.path.exists(tmp_wav_path):
            os.remove(tmp_wav_path)

Evaluating FLEURS id_id:   0%|          | 0/687 [00:00<?, ?it/s]

Evaluating FLEURS id_id: 100%|██████████| 687/687 [4:26:30<00:00, 23.28s/it]  


## METRICS

In [10]:
results_df = pd.DataFrame(detail_rows)
ok_df = results_df[results_df["status"] == "ok"].copy()

if len(ok_df) == 0:
    raise RuntimeError("Tidak ada sample berhasil diproses, jadi metrik tidak bisa dihitung.")

overall_wer_raw = wer(references_raw, predictions_raw)
overall_cer_raw = cer(references_raw, predictions_raw)

overall_wer_clean = wer(references_clean, predictions_clean)
overall_cer_clean = cer(references_clean, predictions_clean)

avg_latency = sum(latencies) / len(latencies)

print("\n=== HASIL EVALUASI ASR ===")
print(f"Model              : {MODEL_NAME}")
print(f"Dataset            : {HF_DATASET_NAME} ({HF_CONFIG_NAME}, split={HF_SPLIT})")
print(f"Jumlah sample      : {len(ds)}")
print(f"Berhasil           : {len(ok_df)}")
print(f"Gagal              : {num_failed}")
print(f"WER (raw)          : {overall_wer_raw:.4f}")
print(f"CER (raw)          : {overall_cer_raw:.4f}")
print(f"WER (cleaned)      : {overall_wer_clean:.4f}")
print(f"CER (cleaned)      : {overall_cer_clean:.4f}")
print(f"Avg latency/file   : {avg_latency:.4f} sec")


=== HASIL EVALUASI ASR ===
Model              : cahya/faster-whisper-medium-id
Dataset            : google/fleurs (id_id, split=test)
Jumlah sample      : 687
Berhasil           : 687
Gagal              : 0
WER (raw)          : 0.2888
CER (raw)          : 0.0680
WER (cleaned)      : 0.0865
CER (cleaned)      : 0.0306
Avg latency/file   : 0.4262 sec


## SAVE FILES

In [11]:
results_df.to_csv(DETAIL_CSV, index=False)

summary_df = pd.DataFrame([{
    "model": MODEL_NAME,
    "dataset_name": HF_DATASET_NAME,
    "dataset_config": HF_CONFIG_NAME,
    "split": HF_SPLIT,
    "num_samples_total": len(ds),
    "num_success": len(ok_df),
    "num_failed": num_failed,
    "wer_raw": overall_wer_raw,
    "cer_raw": overall_cer_raw,
    "wer_cleaned": overall_wer_clean,
    "cer_cleaned": overall_cer_clean,
    "avg_latency_sec": avg_latency,
    "device": DEVICE,
    "compute_type": COMPUTE_TYPE,
    "beam_size": BEAM_SIZE,
    "vad_filter": USE_VAD
}])

summary_df.to_csv(SUMMARY_CSV, index=False)

print("\nSaved files:")
print(f"- {DETAIL_CSV}")
print(f"- {SUMMARY_CSV}")

print("\nContoh hasil:")
display_cols = [
    "id", "reference_text_raw", "pred_text_raw",
    "reference_text_clean", "pred_text_clean",
    "latency_sec", "status"
]
print(results_df[display_cols].head(10).to_string(index=False))


Saved files:
- hasil_evaluasi_fleurs_id_detail.csv
- hasil_evaluasi_fleurs_id_summary.csv

Contoh hasil:
  id                                                                                                                                                                                      reference_text_raw                                                                                                                                                                                                    pred_text_raw                                                                                                                                                                                    reference_text_clean                                                                                                                                                                                           pred_text_clean  latency_sec status
1909                                                      